Deep learning pipeline for image anomaly detection using PyTorch (ImageAnomalyDetection.ipynb v0.1)

*   Train a CNN model (ResNet / DenseNet)
*   Classify images (e.g., Normal vs Abnormal)
*   Evaluate performance (metrics + confusion matrix)
*   Run predictions on new images





In [ ]:
# 1. Install specific requirements
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install Pillow numpy matplotlib seaborn scikit-learn boto3

# 2. Setup paths (AWS equivalent of /content/)
import os
import boto3

# Use the current working directory in SageMaker for persistence
BASE_DIR = os.getcwd()
CACHE_DIR = os.path.join(BASE_DIR, 'cache')
DATASET_DIR = os.path.join(BASE_DIR, 'ImageDataset')

os.makedirs(CACHE_DIR, exist_ok=True)

# 3. Identify the directory for the cache and the dataset

print(f"✅ Environment ready at {BASE_DIR}")
print(f"✅ Cache directory: {CACHE_DIR}")
print(f"✅ Dataset directory: {DATASET_DIR}")

In [ ]:
import torch
import pynvml

def gpu_report():
    print("--- 🖥️ Hardware Diagnostic Report ---")
    
    # 1. Check if CUDA is even available
    cuda_available = torch.cuda.is_available()
    print(f"CUDA Available: {cuda_available}")
    
    if not cuda_available:
        print("❌ No GPU detected. Check your SageMaker instance type.")
        return

    # 2. Get the number of GPUs
    gpu_count = torch.cuda.device_count()
    print(f"Number of GPUs: {gpu_count}")

    # 3. Detailed Memory and Model Info using pynvml
    pynvml.nvmlInit()
    for i in range(gpu_count):
        handle = pynvml.nvmlDeviceGetHandleByIndex(i)
        info = pynvml.nvmlDeviceGetMemoryInfo(handle)
        device_name = pynvml.nvmlDeviceGetName(handle)
        
        print(f"\nGPU {i}: {device_name}")
        print(f"  Total Memory: {info.total / 1024**3:.2f} GB")
        print(f"  Used Memory:  {info.used / 1024**3:.2f} GB")
        print(f"  Free Memory:  {info.free / 1024**3:.2f} GB")
        
        # Current active device check
        if torch.cuda.current_device() == i:
            print(f"  Status: Active Device (Index {i})")

    pynvml.nvmlShutdown()
    print("\n--- End of Report ---")

if __name__ == "__main__":
    gpu_report()

In [ ]:
# Model architecture builder
# Write model.py to file
#
model_code = '''import torch
import torch.nn as nn
from torchvision import models

def build_model(model_name, pretrained=True):
    print(f"Building PyTorch model: {model_name}...")
    if model_name == "ResNet50":
        weights = models.ResNet50_Weights.DEFAULT if pretrained else None
        model = models.resnet50(weights=weights)
        in_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Linear(in_features, 512), nn.ReLU(), nn.Dropout(0.5), nn.Linear(512, 1)
        )
    elif model_name == "DenseNet121":
        weights = models.DenseNet121_Weights.DEFAULT if pretrained else None
        model = models.densenet121(weights=weights)
        in_features = model.classifier.in_features
        model.classifier = nn.Sequential(
            nn.Linear(in_features, 512), nn.ReLU(), nn.Dropout(0.5), nn.Linear(512, 1)
        )
    elif model_name == "EfficientNetB0":
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        model = models.efficientnet_b0(weights=weights)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Sequential(
            nn.Linear(in_features, 512), nn.ReLU(), nn.Dropout(0.5), nn.Linear(512, 1)
        )
    else:
        raise ValueError(f"Unknown model name: {model_name}")

    if pretrained:
        for param in model.parameters():
            param.requires_grad = False
        if model_name == "ResNet50":
            for param in model.fc.parameters(): param.requires_grad = True
        elif model_name == "DenseNet121":
            for param in model.classifier.parameters(): param.requires_grad = True
        elif model_name == "EfficientNetB0":
            for param in model.classifier.parameters(): param.requires_grad = True
    return model'''

with open("model.py", "w") as f:
    f.write(model_code)
print("model.py created.")

In [ ]:
metrics_manager_code = '''import time, json, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import os

class MetricsManager:
    def __init__(self):
        self.start_time = 0
        self.end_time = 0

    def start_timer(self): self.start_time = time.time()
    def stop_timer(self):
        self.end_time = time.time()
        return self.end_time - self.start_time

    def format_time(self, seconds):
        m, s = divmod(seconds, 60)
        h, m = divmod(m, 60)
        return f"{int(h)}h {int(m)}m {int(s)}s" if h else f"{int(m)}m {int(s):.2f}s"

    def calculate_metrics(self, y_true, y_pred_probs):
        y_pred = [1 if p > 0.5 else 0 for p in y_pred_probs]
        return {
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1_score": f1_score(y_true, y_pred, zero_division=0),
            "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
            "report": classification_report(y_true, y_pred, output_dict=True, zero_division=0)
        }

    def plot_confusion_matrix(self, cm, labels=['Normal', 'Anomaly']):
        fig = plt.figure(figsize=(6, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
        plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.tight_layout()
        return fig

    # === UPDATED: Saves raw ROC data + plot (dataset-specific filenames) ===
    # Modified to include model_name in filenames
    def plot_roc_curve(self, y_true, y_probs, dataset_name="unknown", model_name="unknown", save_dir="cache"):
        from sklearn.metrics import roc_curve, auc
        from sklearn.metrics import RocCurveDisplay

        fpr, tpr, _ = roc_curve(y_true, y_probs)
        roc_auc = auc(fpr, tpr)

        # Save raw data for later side-by-side plotting
        roc_data = {"fpr": fpr.tolist(), "tpr": tpr.tolist(), "auc": float(roc_auc)}
        json_path = f"{save_dir}/roc_data_{dataset_name}_{model_name}.json" # Modified line
        with open(json_path, 'w') as f:
            json.dump(roc_data, f)

        # Plot single curve
        display = RocCurveDisplay(fpr=fpr, tpr=tpr, roc_auc=roc_auc)
        display.plot(color='darkorange', lw=2)
        plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random classifier')
        plt.title(f'ROC Curve - {dataset_name} ({model_name}) (AUC = {roc_auc:.4f})') # Modified title
        plt.grid(True, alpha=0.3)
        plt.savefig(f"{save_dir}/roc_curve_{dataset_name}_{model_name}.png", dpi=300, bbox_inches='tight') # Modified line
        plt.close()

        print(f"✅ ROC saved for {dataset_name} ({model_name}) → AUC = {roc_auc:.4f}") # Modified print
        return roc_auc

    def save_training_stats(self, model_name, duration, best_acc, epochs, dataset_name, cache_dir="cache"):
        filepath = os.path.join(cache_dir, "training_stats.json")
        all_stats = {}
        if os.path.exists(filepath):
            try:
                with open(filepath, 'r') as f:
                    all_stats = json.load(f)
            except json.JSONDecodeError:
                print(f"Warning: Could not decode existing {filepath}. Starting with empty stats.")
                all_stats = {}

        current_run_key = f"{model_name}_{dataset_name}"
        all_stats[current_run_key] = {
            "model_name": model_name,
            "duration": duration,
            "best_accuracy": best_acc,
            "epochs": epochs,
            "dataset_name": dataset_name
        }
        with open(filepath, 'w') as f:
            json.dump(all_stats, f, indent=4)
        print(f"📈 Training stats saved for {dataset_name} to {filepath}")
'''

with open("metrics_manager.py", "w") as f:
    f.write(metrics_manager_code)
print("metrics_manager.py updated to include model name in ROC filenames.")

In [ ]:
# Save this as train.py
train_code = '''import os, torch, torch.nn as nn, torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from model import build_model
from metrics_manager import MetricsManager

# 1. Improved Pathing Logic
# During a SageMaker Training Job, SM_CHANNEL_TRAINING is set by AWS.
# For local notebook testing, we use the local path where your S3 data is synced.
# DATASET_DIR = os.environ.get('SM_CHANNEL_TRAINING', '/home/ec2-user/SageMaker/DigitalMediaAnomaly2/ImageDataset')
# MODEL_OUTPUT_DIR = os.environ.get('SM_MODEL_DIR', 'cache')
BASE_DIR = os.getcwd()
CACHE_DIR = os.path.join(BASE_DIR, 'cache')
DATASET_DIR = os.path.join(BASE_DIR, 'ImageDataset')
IMG_SIZE = 150

# os.makedirs(CACHE_DIR, exist_ok=True)

def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_model_pipeline(model_name, epochs=5, batch_size=32, dataset_choice='ucirvine_chest_xray'):
    device = get_device()
    metrics = MetricsManager()

    print(f"🚀 Training {model_name} on {device}...")
    print(f"📁 Dataset Path: {DATASET_DIR}/{dataset_choice}")

    # Safety check for NoneType
    if DATASET_DIR is None or not os.path.exists(DATASET_DIR):
        print(f"❌ ERROR: DATASET_DIR is invalid. Ensure data is synced to {DATASET_DIR}")
        return

    train_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    data_root = os.path.join(DATASET_DIR, dataset_choice)
    train_path = os.path.join(data_root, 'train')
    test_path = os.path.join(data_root, 'test')

    try:
        train_dataset = datasets.ImageFolder(train_path, transform=train_transform)
        val_dataset = datasets.ImageFolder(test_path, transform=val_transform)
    except FileNotFoundError:
        print(f"❌ ERROR: Folders not found in {data_root}. Run S3 sync first.")
        return

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

    model = build_model(model_name).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    metrics.start_timer()
    best_acc = 0.0
    save_path = os.path.join(CACHE_DIR, f"{model_name}_{dataset_choice}_best.pth")

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device).float().unsqueeze(1)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        model.eval()
        correct = total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels_val = images.to(device), labels.to(device).float().unsqueeze(1)
                outputs = model(images)
                preds = torch.sigmoid(outputs) > 0.5
                correct += (preds == labels_val).sum().item()
                total += labels_val.size(0)

        val_acc = correct / total if total > 0 else 0
        print(f"Epoch {epoch+1}/{epochs} | Loss: {running_loss/len(train_loader):.4f} | Val Acc: {val_acc:.2%}")

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), save_path)

    duration = metrics.stop_timer()
    metrics.save_training_stats(model_name, duration, best_acc, epochs, dataset_choice)
    return save_path, best_acc, duration

if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", default="DenseNet121")
    parser.add_argument("--epochs", type=int, default=5)
    parser.add_argument("--batch", type=int, default=32)
    parser.add_argument("--dataset", default="ucirvine_chest_xray")
    args = parser.parse_args()

    train_model_pipeline(args.model, args.epochs, args.batch, args.dataset)
'''

with open("train.py", "w") as f:
    f.write(train_code)
print("✅ train.py updated with local fallback path for DATASET_DIR.")

In [ ]:
anomaly_detection_code = '''import torch
import os
from PIL import Image
from model import build_model
from metrics_manager import MetricsManager
from torchvision import transforms
from train import train_model_pipeline
import numpy as np

# SageMaker Pathing Fixes: Remove all local /content/ or ImageDataset references
# DATASET_DIR points to the S3 bucket mounted locally by SageMaker FastFile
# DATASET_DIR = os.environ.get('SM_CHANNEL_TRAINING') 
# CACHE_DIR points to the designated output folder for artifacts
# CACHE_DIR = os.environ.get('SM_MODEL_DIR', 'cache') 
BASE_DIR = os.getcwd()
CACHE_DIR = os.path.join(BASE_DIR, 'cache')
DATASET_DIR = os.path.join(BASE_DIR, 'ImageDataset')

IMG_SIZE = 150
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(CACHE_DIR, exist_ok=True)

def load_model(model_arch, model_path):
    model = build_model(model_arch, pretrained=False).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    return model

def predict_image(model_path, image_path, model_arch):
    model = load_model(model_arch, model_path)
    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    img = Image.open(image_path).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        prob = torch.sigmoid(output).item()
    label = "ANOMALY" if prob > 0.5 else "NORMAL"
    confidence = prob * 100 if label == "ANOMALY" else (1 - prob) * 100
    return label, confidence

def evaluate_test_set(model_path, dataset_choice='ucirvine_chest_xray', model_arch="DenseNet121"):
    model = load_model(model_arch, model_path)
    
    # Path is now relative to the S3 mount point
    test_dir = os.path.join(DATASET_DIR, dataset_choice, 'test')
    
    metrics_mgr = MetricsManager()
    y_true, y_probs = [], []
    metrics_mgr.start_timer()

    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    for label_idx, category in enumerate(['NORMAL', 'ANOMALY']):
        folder = os.path.join(test_dir, category)
        if not os.path.exists(folder):
            print(f"⚠️ Warning: Folder {folder} not found in S3 mount.")
            continue
        for fname in os.listdir(folder):
            try:
                img = Image.open(os.path.join(folder, fname)).convert('RGB')
                tensor = transform(img).unsqueeze(0).to(device)
                with torch.no_grad():
                    prob = torch.sigmoid(model(tensor)).item()
                y_true.append(label_idx)
                y_probs.append(prob)
            except Exception:
                continue

    duration = metrics_mgr.stop_timer()
    if not y_true:
        print(f"❌ No images found in {test_dir}. Check S3 bucket structure.")
        return None

    results = metrics_mgr.calculate_metrics(y_true, y_probs)
    roc_auc = metrics_mgr.plot_roc_curve(y_true, y_probs, dataset_name=dataset_choice, model_name=model_arch, save_dir=CACHE_DIR)
    results["roc_auc"] = float(roc_auc)

    print(f"✅ Evaluation complete in {duration:.1f}s | AUC-ROC: {results['roc_auc']:.4f}")
    fig = metrics_mgr.plot_confusion_matrix(np.array(results['confusion_matrix']))
    fig.savefig(f"{CACHE_DIR}/confusion_matrix_{dataset_choice}_{model_arch}.png")
    return results

if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--mode", choices=["train", "predict", "evaluate"], required=True)
    parser.add_argument("--model", default="DenseNet121", choices=["DenseNet121", "ResNet50", "EfficientNetB0"])
    parser.add_argument("--epochs", type=int, default=5)
    parser.add_argument("--batch", type=int, default=32)
    parser.add_argument("--dataset", default="ucirvine_chest_xray")
    parser.add_argument("--image", help="Path to image for prediction")
    parser.add_argument("--model_path", help="Path to .pth model")
    args = parser.parse_args()

    if args.mode == "train":
        train_model_pipeline(args.model, args.epochs, args.batch, args.dataset)
    elif args.mode == "predict":
        if not args.image or not args.model_path:
            print("❌ Provide --image and --model_path")
        else:
            label, confidence = predict_image(args.model_path, args.image, args.model)
            print(f"RESULT: {label} ({confidence:.1f} %)")
    elif args.mode == "evaluate":
        if not args.model_path:
            print("❌ Provide --model_path")
        else:
            evaluate_test_set(args.model_path, args.dataset, args.model)
'''

with open("anomaly_detection.py", "w") as f:
    f.write(anomaly_detection_code)
print("anomaly_detection.py updated to pass model name to ROC and confusion matrix saving functions.")

In [ ]:
# @title
# Example training (models available: DenseNet121, EfficientNetB0, and ResNet50 )
!python anomaly_detection.py --mode train --model DenseNet121 --epochs 5 --dataset ucirvine_chest_xray
!python anomaly_detection.py --mode train --model ResNet50 --epochs 5 --dataset ucirvine_chest_xray
!python anomaly_detection.py --mode train --model EfficientNetB0 --epochs 5 --dataset ucirvine_chest_xray

# Example single prediction
# !python anomaly_detection.py --mode predict --model_path /content/cache/DenseNet121_best.pth --image /content/drive/MyDrive/ImageDataset/ucirvine_chest_xray/val/NORMAL/NORMAL2-IM-1427-0001.jpeg

# Example full evaluation
#!python anomaly_detection.py --model DenseNet121 --mode evaluate --model_path /content/cache/DenseNet121_best.pth --dataset yonsei_faces

In [ ]:
!python anomaly_detection.py --mode evaluate --model DenseNet121 --model_path cache/DenseNet121_ucirvine_chest_xray_best.pth --dataset ucirvine_chest_xray
!python anomaly_detection.py --mode evaluate --model ResNet50 --model_path cache/ResNet50_ucirvine_chest_xray_best.pth --dataset ucirvine_chest_xray
!python anomaly_detection.py --mode evaluate --model EfficientNetB0 --model_path cache/EfficientNetB0_ucirvine_chest_xray_best.pth --dataset ucirvine_chest_xray

In [ ]:
from IPython.display import Image, display
import matplotlib.pyplot as plt
import os
import json
import math

def get_epochs_from_stats(model_name, dataset_name, cache_dir="cache"):
    filepath = os.path.join(cache_dir, "training_stats.json")
    if os.path.exists(filepath):
        try:
            with open(filepath, 'r') as f:
                all_stats = json.load(f)
            key = f"{model_name}_{dataset_name}"
            if key in all_stats and "epochs" in all_stats[key]:
                return all_stats[key]["epochs"]
        except json.JSONDecodeError:
            print(f"Warning: Could not decode existing {filepath}.")
    return None

def plot_multi_model_confusion_matrices(models, dataset_name="ucirvine_chest_xray", cache_dir="cache"):
    num_models = len(models)
    # Calculate grid dimensions for wrapping after 2 graphs horizontally
    ncols = 2
    nrows = math.ceil(num_models / ncols)

    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 6 * nrows))

    # Flatten axes array for easier iteration if it's a 2D array
    if nrows > 1 or ncols > 1:
        axes = axes.flatten()
    else:
        axes = [axes]

    for i, model_name in enumerate(models):
        ax = axes[i]
        img_path = os.path.join(cache_dir, f"confusion_matrix_{dataset_name}_{model_name}.png")
        if os.path.exists(img_path):
            img = plt.imread(img_path)
            ax.imshow(img)
            ax.axis('off')
            ax.set_title(f'{model_name}', fontsize=14)
        else:
            print(f"Warning: Confusion matrix image not found for {model_name} on {dataset_name} at {img_path}")
            ax.axis('off')
            ax.set_title(f'{model_name} (Not Found)', fontsize=14)

    # Hide any unused subplots
    for j in range(num_models, len(axes)):
        fig.delaxes(axes[j])

    # Dynamically get epochs
    epochs = get_epochs_from_stats(models[0], dataset_name) if models else None
    epochs_str = f" ({epochs} epochs)" if epochs is not None else ""

    plt.suptitle(f'Confusion Matrix Comparison on {dataset_name}{epochs_str}', fontsize=16, y=1.02)
    plt.tight_layout(rect=[0, 0, 1, 0.98]) # Adjust layout to prevent suptitle overlap
    save_path = os.path.join(cache_dir, f"confusion_matrix_comparison_{dataset_name}.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✅ Multi-model Confusion Matrix comparison saved as {save_path}")

models_to_compare = ["DenseNet121", "ResNet50", "EfficientNetB0"]
plot_multi_model_confusion_matrices(models_to_compare, dataset_name="ucirvine_chest_xray")

In [ ]:
import json
import matplotlib.pyplot as plt
import os
from sklearn.metrics import RocCurveDisplay
import math

def get_epochs_from_stats(model_name, dataset_name, cache_dir="cache"):
    filepath = os.path.join(cache_dir, "training_stats.json")
    if os.path.exists(filepath):
        try:
            with open(filepath, 'r') as f:
                all_stats = json.load(f)
            key = f"{model_name}_{dataset_name}"
            if key in all_stats and "epochs" in all_stats[key]:
                return all_stats[key]["epochs"]
        except json.JSONDecodeError:
            print(f"Warning: Could not decode existing {filepath}.")
    return None

def plot_separate_roc_curves(models, dataset_name="ucirvine_chest_xray", cache_dir="cache"):
    num_models = len(models)
    # Calculate grid dimensions for wrapping after 2 graphs horizontally
    ncols = 2
    nrows = math.ceil(num_models / ncols)

    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 6 * nrows))

    # Flatten axes array for easier iteration if it's a 2D array
    if nrows > 1 or ncols > 1:
        axes = axes.flatten()
    else:
        axes = [axes]

    colors = ['darkorange', 'blue', 'green'] # Define colors for consistency

    for i, model_name in enumerate(models):
        ax = axes[i]
        json_path = os.path.join(cache_dir, f"roc_data_{dataset_name}_{model_name}.json")
        if os.path.exists(json_path):
            with open(json_path, 'r') as f:
                data = json.load(f)
            fpr = data["fpr"]
            tpr = data["tpr"]
            roc_auc = data["auc"]

            ax.plot(fpr, tpr, color=colors[i % len(colors)], lw=2,
                    label=f'{model_name} (AUC = {roc_auc:.4f})')
            ax.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random classifier')
            ax.set_xlim([0.0, 1.0])
            ax.set_ylim([0.0, 1.05])
            ax.set_xlabel('False Positive Rate')
            ax.set_ylabel('True Positive Rate')
            ax.set_title(f'ROC Curve - {model_name}\n(AUC = {roc_auc:.4f})')
            # ax.legend(loc="lower right") # Removed legend as requested
            ax.grid(True, alpha=0.3)
        else:
            ax.text(0.5, 0.5, f"No data for {model_name}", horizontalalignment='center', verticalalignment='center', transform=ax.transAxes)
            ax.set_title(f'ROC Curve - {model_name} (No Data)')
            print(f"Warning: ROC data not found for {model_name} on {dataset_name} at {json_path}")

    # Hide any unused subplots
    for j in range(num_models, len(axes)):
        fig.delaxes(axes[j])

    epochs = get_epochs_from_stats(models[0], dataset_name) if models else None
    epochs_str = f" ({epochs} epochs)" if epochs is not None else ""

    fig.suptitle(f'ROC Curves Comparison on {dataset_name}{epochs_str}', fontsize=16, y=1.02)
    plt.tight_layout(rect=[0, 0, 1, 0.98])
    save_path = os.path.join(cache_dir, f"roc_curves_comparison_separate_{dataset_name}.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✅ Multi-model ROC curves comparison (separate plots) saved as {save_path}")

models_to_compare = ["DenseNet121", "ResNet50", "EfficientNetB0"]
plot_separate_roc_curves(models_to_compare, dataset_name="ucirvine_chest_xray")